In [ ]:
import sys
lib_path = [r'C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisRoutine',
            r'C:\Users\ikahb\OneDrive\Applications\GitHub\SeisRoutine']
for path in lib_path:
    sys.path.append(path)
##########################################################################
import SeisRoutine.config as srconf
import SeisRoutine.seisbench as srsb

In [ ]:
import seisbench.generate as sbg
import seisbench.data as sbd

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
timestamp = srconf.timestamp()
cfg = srconf.Config.load(
    file_path='./Configs/Parameters-cfg.yml',
    resolve=True,
)
context={
    "timestamp": timestamp,
    "project": "Check Channel Replacement",
}
cfg.resolve(context=context)

In [ ]:
data_format = cfg.to_dict()['dataset']['data_format']
data_format_tmp = data_format.copy()
dataset = sbd.WaveformDataset(
    path=cfg.dataset.path,
    **data_format_tmp
)

dataset.metadata['channel_status'] = [[np.random.randint(0, 3) for jj in range(3)] for ii in range(81015)]

phase_dict = srsb.dataset.build_phase_mapper(
    dataset.metadata.columns
)
context = {
    "phase_dict_keys": list(phase_dict.keys()),
    "phase_dict": phase_dict,
    "np": np
}
cfg.resolve(context=context)

augmentations = srsb.dataset.build_augmentations(cfg.augmentation)
generator = srsb.dataset.make_generator(dataset, augmentations)

In [ ]:
generator[0].keys()

In [ ]:
cond_PS_pairs = srsb.dataset.find_ps_pairs(metadata=dataset.metadata)

In [ ]:
n = [32]
n = [18, 30,37,76]
# n = [18, 30,37,76,87,97,105,109,126,137,149,158,161,173,179,196,204,209, 217,
#      222, 230,235,251,254,257,261,260,265,266,267,268,273,275,290,302,310,
#      312,323,331,335,355,367,376,388,391,393]
for ii in n:
    metadata = dataset.metadata.iloc[ii]
    print(ii, metadata['channel_status'])
    data = generator[ii]
    data_X = data['X']
    data_X0 = data['X_io_BadChannelReplacer']
    data_X0 = data_X0 / data_X0.max(axis=1).reshape(-1, 1)
    ###
    fig, axes = plt.subplots(1, 2,
        figsize=(10, 3))
    label = [_ for _ in dataset.component_order]
    axes[0].plot(data_X.T +[1, 0, -1], label=label)
    axes[1].plot(data_X0.T+[1, 0, -1], label=label)
    plt.legend(loc=1)
    plt.show()
    # print(cond_all.iloc[ii])
    # print(df_channel_condition.iloc[ii], cond[ii])

In [ ]:
metadata = dataset.metadata.iloc[32]

with pd.option_context('display.max_rows', None):
    print(metadata, type(metadata))